Notebook này thực hiện kiến trúc Medallion (Bronze $\rightarrow$ Silver $\rightarrow$ Gold) bằng PySpark trên Google Colab, sau đó đẩy thẳng bộ dữ liệu Gold sạch lên Hugging Face Dataset Hub để Backend Server có thể tải về bất cứ lúc nào.Python

# Start
## CÀI ĐẶT THƯ VIỆN & KHỞI TẠO MÔI TRƯỜNG (GOOGLE COLAB)

In [1]:
# ==============================================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN & KHỞI TẠO MÔI TRƯỜNG (GOOGLE COLAB)
# ==============================================================================
# !pip install -q pyspark kagglehub huggingface_hub pyarrow pandas

import os
import re
import kagglehub
from huggingface_hub import HfApi, login
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType
from google.colab import userdata

# Cấu hình Hugging Face (Thay bằng Token và Repo của bạn)
# HF_TOKEN = os.getenv("HF_TOKEN", "hf_YOUR_HUGGINGFACE_WRITE_TOKEN")
HF_TOKEN = userdata.get('HF_TOKEN')
HF_DATASET_REPO = "cuongduongok/yt-trend-forecaster-gold-data" # Repo loại Dataset trên HF

# Khởi tạo SparkSession tối ưu cho RAM của Google Colab (12GB)
spark = SparkSession.builder \
    .appName("YTTrendForecaster_Medallion_ETL") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.LegacyTimeParserPolicy", "LEGACY") \
    .getOrCreate()

print("✅ Spark Session đã sẵn sàng! Phiên bản:", spark.version)

✅ Spark Session đã sẵn sàng! Phiên bản: 4.0.4


## TẦNG BRONZE (INGESTION) - TẢI DỮ LIỆU PARQUET TỪ KAGGLEHUB

In [2]:
# ==============================================================================
# CELL 2: TẦNG BRONZE (INGESTION) - TẢI DỮ LIỆU PARQUET TỪ KAGGLEHUB
# ==============================================================================
print("⏳ Đang tải dataset từ Kaggle qua kagglehub...")
dataset_dir = kagglehub.dataset_download("canerkonuk/youtube-trending-videos-global")
parquet_path = os.path.join(dataset_dir, "youtube_trending_videos_global.parquet")

# Đọc file Parquet vào PySpark DataFrame
df_bronze = spark.read.parquet(parquet_path)
print(f"✅ Đã nạp tầng Bronze! Tổng số bản ghi thô: {df_bronze.count():,}")
df_bronze.select("video_id", "video_trending__date", "video_trending_country", "video_category_id", "video_view_count").show(5)

⏳ Đang tải dataset từ Kaggle qua kagglehub...


KaggleApiHTTPError: 404 Client Error.

Resource not found at URL: https://kaggle.com/datasets/canerkonuk/youtube-trending-videos-global/versions/742
Please make sure you specified the correct resource identifiers.

Hoặc

In [3]:
import os
import kagglehub

# Tải riêng file parquet thay vì nén toàn bộ dataset
parquet_path = kagglehub.dataset_download(
    "canerkonuk/youtube-trending-videos-global",
    path="youtube_trending_videos_global.parquet"
)

# Đọc file Parquet vào PySpark DataFrame
df_bronze = spark.read.parquet(parquet_path)
print(f"✅ Đã nạp tầng Bronze! Tổng số bản ghi thô: {df_bronze.count():,}")
df_bronze.select("video_id", "video_trending__date", "video_trending_country", "video_category_id", "video_view_count").show(5)

100%|██████████| 4.30G/4.30G [01:05<00:00, 70.8MB/s]


✅ Đã nạp tầng Bronze! Tổng số bản ghi thô: 12,438,165
+-----------+--------------------+----------------------+-----------------+----------------+
|   video_id|video_trending__date|video_trending_country|video_category_id|video_view_count|
+-----------+--------------------+----------------------+-----------------+----------------+
|bB3-CUMERIU|          2024.10.12|  United Arab Emirates|            Music|      20535235.0|
|5ObJt_71AYc|          2024.10.12|  United Arab Emirates|           Sports|       3966042.0|
|zfb0whgBBA8|          2024.10.12|  United Arab Emirates|           Gaming|        853167.0|
|SJfoPdeOPCQ|          2024.10.12|  United Arab Emirates|           Sports|       3758707.0|
|UVb6QOKy0bI|          2024.10.12|  United Arab Emirates| Film & Animation|       1730189.0|
+-----------+--------------------+----------------------+-----------------+----------------+
only showing top 5 rows


In [4]:
# Xem danh sách tất cả các cột và kiểu dữ liệu
df_bronze.printSchema()

# Xem tổng số lượng cột
print("Số lượng cột:", len(df_bronze.columns))

root
 |-- video_id: string (nullable = true)
 |-- video_published_at: string (nullable = true)
 |-- video_trending__date: string (nullable = true)
 |-- video_trending_country: string (nullable = true)
 |-- channel_id: string (nullable = true)
 |-- video_title: string (nullable = true)
 |-- video_description: string (nullable = true)
 |-- video_default_thumbnail: string (nullable = true)
 |-- video_category_id: string (nullable = true)
 |-- video_tags: string (nullable = true)
 |-- video_duration: string (nullable = true)
 |-- video_dimension: string (nullable = true)
 |-- video_definition: string (nullable = true)
 |-- video_licensed_content: string (nullable = true)
 |-- video_view_count: string (nullable = true)
 |-- video_like_count: string (nullable = true)
 |-- video_comment_count: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- channel_description: string (nullable = true)
 |-- channel_custom_url: string (nullable = true)
 |-- channel_published_at: stri

## TẦNG SILVER (CLEANING, DEDUPLICATION & TAG EXTRACTION)

In [9]:
import pyspark.sql.functions as F

# 1. Tắt ANSI mode để tránh crash khi parse ngày/số không hợp lệ
spark.conf.set("spark.sql.ansi.enabled", "false")

# ==============================================================================
# CELL 3: TẦNG SILVER (CLEANING, DEDUPLICATION & TAG EXTRACTION)
# ==============================================================================

# Chuẩn hóa chuỗi ngày: lấy 10 ký tự đầu và thay '.' thành '-'
raw_date_clean = F.regexp_replace(F.substring(F.trim(F.col("video_trending__date")), 1, 10), r"\.", "-")

df_silver_base = df_bronze \
    .filter(
        F.col("video_id").isNotNull() &
        F.col("video_trending__date").isNotNull() &
        F.col("video_category_id").isNotNull()
    ) \
    .withColumn("clean_date_str", raw_date_clean) \
    .withColumn(
        "trending_date",
        # Chỉ parse nếu đúng pattern YYYY-MM-DD (4 số - 2 số - 2 số)
        F.when(
            F.col("clean_date_str").rlike(r"^\d{4}-\d{2}-\d{2}$"),
            F.to_date(F.col("clean_date_str"), "yyyy-MM-dd")
        ).otherwise(F.lit(None))
    ) \
    .filter(F.col("trending_date").isNotNull()) \
    .drop("clean_date_str") \
    .withColumn("view_count", F.coalesce(F.col("video_view_count").cast("double").cast("long"), F.lit(0))) \
    .withColumn("like_count", F.coalesce(F.col("video_like_count").cast("double").cast("long"), F.lit(0))) \
    .withColumn("comment_count", F.coalesce(F.col("video_comment_count").cast("double").cast("long"), F.lit(0))) \
    .withColumn("clean_tags", F.lower(F.trim(F.coalesce(F.col("video_tags"), F.lit("")))))

# Bù khuyết Tags bằng chính Category nếu video_tags rỗng
df_silver_enriched = df_silver_base.withColumn(
    "effective_tags_array",
    F.when(
        F.length(F.col("clean_tags")) > 0,
        F.split(F.col("clean_tags"), r"\s*,\s*")
    ).otherwise(
        F.array(F.lower(F.trim(F.col("video_category_id"))))
    )
)

# Gom nhóm chống trùng lặp Video trong cùng 1 ngày (Global Video-Day Deduplication)
df_silver_unique_video_day = df_silver_enriched.groupBy("video_id", "trending_date", "video_category_id").agg(
    F.max("view_count").alias("daily_global_views"),
    F.max("like_count").alias("daily_global_likes"),
    F.max("comment_count").alias("daily_global_comments"),
    F.countDistinct("video_trending_country").alias("trending_countries_count"),
    F.first("effective_tags_array").alias("tags_array")
)

print("✅ Đã xử lý xong tầng Silver (Chống nhân bản view đa quốc gia & Bù khuyết Tags bằng Category)!")
df_silver_unique_video_day.show(5)

✅ Đã xử lý xong tầng Silver (Chống nhân bản view đa quốc gia & Bù khuyết Tags bằng Category)!
+-----------+-------------+-----------------+------------------+------------------+---------------------+------------------------+--------------------+
|   video_id|trending_date|video_category_id|daily_global_views|daily_global_likes|daily_global_comments|trending_countries_count|          tags_array|
+-----------+-------------+-----------------+------------------+------------------+---------------------+------------------------+--------------------+
|--6qfzROsOQ|   2024-10-31|    Howto & Style|            159967|             10208|                  474|                       1|[pollo con papas,...|
|--6qfzROsOQ|   2024-11-04|    Howto & Style|            176532|             10917|                  477|                       1|[pollo con papas,...|
|--8REpvd5lo|   2026-09-01|           Gaming|             73432|               245|                    1|                       3|            [gam

In [10]:
df_silver_unique_video_day.cache()

DataFrame[video_id: string, trending_date: date, video_category_id: string, daily_global_views: bigint, daily_global_likes: bigint, daily_global_comments: bigint, trending_countries_count: bigint, tags_array: array<string>]

## TẦNG GOLD (TIME-SERIES AGGREGATION THEO CATEGORY VÀ TOP TAGS)

In [11]:
# ==============================================================================
# CELL 4: TẦNG GOLD (TIME-SERIES AGGREGATION THEO CATEGORY VÀ TOP TAGS)
# ==============================================================================
# Ta tạo chuỗi thời gian cho cả 2 cấp độ:
# 1. Cấp độ Danh mục (CATEGORY): Music, Gaming, Entertainment, Sports...
# 2. Cấp độ Từ khóa ngách (TAG): Top 100 từ khóa có tổng lượng xuất hiện lớn nhất.

# --- 4.1. Chuỗi thời gian theo CATEGORY ---
df_gold_category = df_silver_unique_video_day.groupBy(
    F.col("trending_date").alias("ds"),
    F.col("video_category_id").alias("topic_name")
).agg(
    F.lit("CATEGORY").alias("topic_type"),
    F.sum("daily_global_views").alias("total_views"),
    F.sum("daily_global_likes").alias("total_likes"),
    F.sum("daily_global_comments").alias("total_comments"),
    F.countDistinct("video_id").alias("trending_video_count"),
    F.sum("trending_countries_count").alias("total_country_appearances")
)

# --- 4.2. Chuỗi thời gian theo TOP 100 TAGS ---
df_exploded_tags = df_silver_unique_video_day \
    .select("video_id", "trending_date", "daily_global_views", "daily_global_likes", "daily_global_comments", "trending_countries_count", F.explode("tags_array").alias("tag")) \
    .withColumn("tag", F.trim(F.col("tag"))) \
    .filter((F.length(F.col("tag")) >= 3) & (F.length(F.col("tag")) <= 30))

# Tìm Top 100 tags phổ biến nhất toàn tập dữ liệu để đảm bảo chuỗi thời gian đủ dày
top_100_tags = [row["tag"] for row in df_exploded_tags.groupBy("tag").count().orderBy(F.desc("count")).limit(100).collect()]

df_gold_tags = df_exploded_tags.filter(F.col("tag").isin(top_100_tags)).groupBy(
    F.col("trending_date").alias("ds"),
    F.col("tag").alias("topic_name")
).agg(
    F.lit("TAG").alias("topic_type"),
    F.sum("daily_global_views").alias("total_views"),
    F.sum("daily_global_likes").alias("total_likes"),
    F.sum("daily_global_comments").alias("total_comments"),
    F.countDistinct("video_id").alias("trending_video_count"),
    F.sum("trending_countries_count").alias("total_country_appearances")
)

# Hợp nhất 2 bảng thành bảng Gold Time-Series duy nhất
df_gold_timeseries = df_gold_category.unionByName(df_gold_tags) \
    .withColumn("engagement_rate", F.round((F.col("total_likes") + F.col("total_comments")) / F.when(F.col("total_views") > 0, F.col("total_views")).otherwise(1), 6)) \
    .orderBy("topic_type", "topic_name", "ds")

print(f"✅ Hoàn tất tầng Gold! Tổng số điểm dữ liệu chuỗi thời gian: {df_gold_timeseries.count():,}")
df_gold_timeseries.show(10)

✅ Hoàn tất tầng Gold! Tổng số điểm dữ liệu chuỗi thời gian: 80,726
+----------+----------------+----------+-----------+-----------+--------------+--------------------+-------------------------+---------------+
|        ds|      topic_name|topic_type|total_views|total_likes|total_comments|trending_video_count|total_country_appearances|engagement_rate|
+----------+----------------+----------+-----------+-----------+--------------+--------------------+-------------------------+---------------+
|2024-10-12|Autos & Vehicles|  CATEGORY|   62670000|    2634696|        158124|                  81|                      183|       0.044564|
|2024-10-13|Autos & Vehicles|  CATEGORY|   61494957|    2564864|        158784|                  84|                      179|       0.044291|
|2024-10-14|Autos & Vehicles|  CATEGORY|   65036177|    2677800|        158520|                  83|                      177|       0.043611|
|2024-10-15|Autos & Vehicles|  CATEGORY|   68684609|    2839793|        170

## XUẤT PARQUET SẠCH & ĐẨY LÊN HUGGING FACE DATASET HUB

In [12]:
import os
import pandas as pd
from huggingface_hub import HfApi, login

# ==============================================================================
# CELL 5: XUẤT PARQUET SẠCH & ĐẨY LÊN HUGGING FACE DATASET HUB
# ==============================================================================
os.makedirs("gold_output", exist_ok=True)
local_gold_parquet = "gold_output/gold_topic_timeseries.parquet"

# Chuyển về Pandas để xuất ra 1 file Parquet duy nhất gọn nhẹ cho Server dễ tải
pdf_gold = df_gold_timeseries.toPandas()

# Chuẩn hóa cột ds thành datetime chuẩn của Pandas (rất quan trọng cho Prophet sau này)
pdf_gold["ds"] = pd.to_datetime(pdf_gold["ds"])

# Xuất Parquet với nén Snappy tối ưu dung lượng
pdf_gold.to_parquet(local_gold_parquet, index=False, engine="pyarrow")
print(f"📦 Đã lưu file local: {local_gold_parquet} ({os.path.getsize(local_gold_parquet) / (1024*1024):.2f} MB)")

# Đẩy lên Hugging Face Hub
if HF_TOKEN and HF_TOKEN != "hf_YOUR_HUGGINGFACE_WRITE_TOKEN":
    login(token=HF_TOKEN)
    api = HfApi()
    api.create_repo(repo_id=HF_DATASET_REPO, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=local_gold_parquet,
        path_in_repo="gold_topic_timeseries.parquet",
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        commit_message="Update Gold Time-Series Dataset from PySpark ETL"
    )
    print(f"🚀 Đã đẩy thành công Dataset sạch lên Hugging Face: https://huggingface.co/datasets/{HF_DATASET_REPO}")
else:
    print("⚠️ Chưa cấu hình HF_TOKEN hợp lệ. File Parquet đã được lưu sẵn tại:", local_gold_parquet)

📦 Đã lưu file local: gold_output/gold_topic_timeseries.parquet (2.28 MB)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._topic_timeseries.parquet:  22%|##2       |  534kB / 2.39MB            

Validating                    :           |   0%            

🚀 Đã đẩy thành công Dataset sạch lên Hugging Face: https://huggingface.co/datasets/cuongduongok/yt-trend-forecaster-gold-data
